In [96]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import make_scorer, accuracy_score, f1_score
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from imblearn.combine import SMOTEENN

# Suport functions

In [97]:
def confusion(true, pred):
    """
    Function for pretty printing confusion matrices
    """
    true.name = 'target'
    pred.name = 'predicted'
    cm = pd.crosstab(true.reset_index(drop=True), pred.reset_index(drop=True))
    cm = cm[cm.index]
    return cm

# Data loading

In [98]:
ILDS = pd.read_csv("minimal_train_fs.csv", delimiter=',', header = None)

ILDS.columns = ['Age', 'TB', 'Alkphos', 'Sgot', 'ALB', 'AR', 'BilRatio', 'Female', 'Target']


display(ILDS)

,Age,TB,Alkphos,Sgot,ALB,AR,BilRatio,Female,Target
0,48,1.504077,5.641907,4.304065,2.4,0.52,0.511111,0,0
1,39,0.641854,5.192957,4.127134,4.3,1.38,0.473684,0,0
2,23,0.000000,5.356586,4.382027,3.1,1.00,0.300000,0,0
3,42,-0.356675,5.023881,4.394449,3.2,1.06,0.285714,1,0
4,54,3.117950,6.324359,3.610918,3.4,0.80,0.504425,1,0
...,...,...,...,...,...,...,...,...,...
444,38,-0.223144,4.976734,3.135494,3.1,1.03,0.250000,1,1
445,63,-0.105361,5.267858,3.806662,3.9,1.85,0.222222,0,1
446,60,-0.356675,5.141664,3.258097,3.5,1.00,0.285714,0,1
447,35,-0.105361,5.247024,2.995732,3.6,1.20,0.222222,0,1


In [99]:
X = ILDS.loc[:, ILDS.columns != 'Target']
y = ILDS['Target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify = y, random_state=1234)

In [100]:
from imblearn.under_sampling import RandomUnderSampler

# Define the undersampler
rus = RandomUnderSampler(random_state=123454)

# Apply it to the training data
X_train, y_train = rus.fit_resample(X_train, y_train)


In [101]:
# from imblearn.over_sampling import RandomOverSampler

# ros = RandomOverSampler(random_state=1234)
# X_train, y_train = ros.fit_resample(X_train, y_train)


In [102]:
# from imblearn.over_sampling import SMOTE

# smote = SMOTE(random_state=1234)
# X_train, y_train = smote.fit_resample(X_train, y_train)


In [103]:
results_df = pd.DataFrame(index=[], columns= ['Accuracy', 'F1 Macro', 'Precision Macro', 'Recall Macro'])

# Decision tree

# Random forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=1234, class_weight='balanced')
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

In [105]:
confusion(y_train, pd.Series(rf.predict(X_train)))

predicted,0,1
target,,
0,101,0
1,0,101


In [106]:
confusion(y_test, pd.Series(rf.predict(X_test)))     

predicted,0,1
target,,
0,45,20
1,7,18


In [107]:
cross_val_results = pd.DataFrame(cross_validate(rf , X_train, y_train, cv = 5, 
                            scoring = ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro'] ))

results_df.loc['RF',:] = cross_val_results[['test_accuracy', 'test_f1_macro',
       'test_precision_macro', 'test_recall_macro']].mean().values
results_df

,Accuracy,F1 Macro,Precision Macro,Recall Macro
RF,0.702439,0.699836,0.706686,0.702381


# F1 Check

In [108]:
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
import numpy as np

def compute_metrics (y_real, y_pred) -> list[float]:
    F1_macro = f1_score(y_real, y_pred, average = "macro")
    recall = recall_score(y_real, y_pred, average = "macro")
    prec = precision_score(y_real, y_pred, average = "macro")
    acc = accuracy_score(y_real, y_pred)
    return [F1_macro, recall, prec, acc]

def confusion (y_real, y_pred) -> None:
    TP = sum(np.logical_and(y_real == y_pred, y_real == 1))
    TN = sum(np.logical_and(y_real == y_pred, y_real == 0))
    FP = sum(np.logical_and(y_real != y_pred, y_real == 0))
    FN = sum(np.logical_and(y_real != y_pred, y_real == 1))
    print("\t\tPredicted")
    print("\t\t+1\t0")
    print(f"Real\t+1\t{TP}\t{FN}")
    print(f"\t0\t{FP}\t{TN}")
    print(f"Accuracy: {((TP + TN) / y_real.shape[0] * 100):.2f}%".format())

metrics_df = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])

In [109]:
test_y = pd.read_csv("test_y.csv").iloc[:, 1]

ILDS_test = pd.read_csv("minimal_test_fs.csv", delimiter=',', header = None)

ILDS_test.columns = ['Age', 'TB', 'Alkphos', 'Sgot', 'ALB', 'AR', 'BilRatio', 'Female']

random forest: rf

gradient boosting: gb

voting classifier: voting

xgboost: xgb

ada boosting: ada

In [110]:
from sklearn.ensemble import RandomForestClassifier

rf.fit(X_train, y_train)

rf.fit(X_train, y_train)

labels_rf = pd.DataFrame(columns = ['ID', 'Label'])
labels_rf['Label'] = pd.DataFrame(rf.predict(ILDS_test))
labels_rf['ID'] = labels_rf.index + 1

confusion(test_y, labels_rf['Label'])

compute_metrics(test_y, labels_rf['Label'])

		Predicted
		+1	0
Real	+1	26	7
	0	25	58
Accuracy: 72.41%


[0.7014157014157014,
 0.7433369843008397,
 0.7010558069381598,
 0.7241379310344828]

# Final test export

In [111]:
ILDS_test = pd.read_csv("minimal_test_fs.csv", delimiter=',', header = None)

ILDS_test.columns = ['Age', 'TB', 'Alkphos', 'Sgot', 'ALB', 'AR', 'BilRatio', 'Female']

X_test = ILDS_test.loc[:,:'Female']

ILDS_test['Label'] = rf.predict(X_test)


ILDS_test.index = ILDS_test.index + 1
ILDS_test.index.name = 'ID'

ILDS_test['Label'].to_csv('random_forest_fs.csv', index=True)